# Audition model outputs — hear & see predicted drum dynamics

Pick a drum performance from E-GMD, run a trained model to predict every note's
**velocity** from structure/timing only, then listen to it and view its piano roll
next to the original human performance.

**How to use**
1. Set `MODEL` / `SPLIT` below and run the setup cells **once** (loading the model is
   the slow part).
2. Call `audition(N)` to hear/see the N-th sample. **To change the sample, just edit
   the number and re-run that one cell** — no reload. `audition()` picks a random one.
3. Or drag the slider in the interactive cell at the bottom.

**Requirements**: run with the repo `.venv` kernel; needs `fluidsynth` and the
soundfont at `sf/big/FluidR3_GM.sf2`.

> ⚠️ **When you change `MODEL`, restart the kernel and re-run.** LightGBM and PyTorch
> each load their own OpenMP runtime; mixing both in one kernel can crash on macOS.

Prereqs from the training scripts: `data/processed/lightgbm_model.joblib`,
`data/processed/transformer_best.pt`, `data/processed/transformer_meta.json`.

In [ ]:
# ── Config (model-level; run the setup cells once after changing these) ────
MODEL = "transformer"     # "transformer" (Plan B) or "lightgbm" (Plan A)
SPLIT = "test"            # default split for audition(): "train" | "validation" | "test"

In [ ]:
# ── Setup: paths, light imports, dataset index ────────────────────────────
import os, sys, warnings
sys.path.insert(0, "..")          # make the drumhumanizer package importable from notebooks/
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Only torch-free helpers here; the model-specific stack loads in the next cell.
from drumhumanizer.midi import load_note_array
from drumhumanizer.features import build_note_features
from drumhumanizer.metrics import mae, rmse
from drumhumanizer.playback import play_midi_file, play_midi_notes, set_soundfont
from drumhumanizer.viz import drums_roll

# repo-relative locations (cwd is notebooks/)
BASE = os.path.join("..", "data", "e-gmd", "e-gmd-v1.0.0")
PROC = os.path.join("..", "data", "processed")
set_soundfont(os.path.join("..", "sf", "big", "FluidR3_GM.sf2"))

csv = pd.read_csv(os.path.join(BASE, "e-gmd-v1.0.0.csv"))
print(f"dataset rows per split: {csv.split.value_counts().to_dict()}")

## Load the selected model (once)

Builds a `predict(feats) -> velocities` function for `MODEL`. Only the selected
model's stack is imported (torch **or** lightgbm, never both in one kernel).

In [ ]:
if MODEL == "lightgbm":
    import joblib
    _saved = joblib.load(os.path.join(PROC, "lightgbm_model.joblib"))

    def predict(feats):
        X = feats.drop(columns=_saved["drop"])
        for c in _saved["cat"]:                       # align categoricals to train levels
            X[c] = X[c].astype("category").cat.set_categories(_saved["cat_categories"][c])
        return _saved["model"].predict(X, num_iteration=_saved["best_iteration"])

elif MODEL == "transformer":
    import json
    import torch
    from drumhumanizer.model import VelocityTransformer
    from drumhumanizer.seqdata import build_split_tensors, scatter_predictions
    _sc = json.load(open(os.path.join(PROC, "transformer_meta.json")))
    _gv = {k: int(v) for k, v in _sc["genre_vocab"].items()}
    _ck = torch.load(os.path.join(PROC, "transformer_best.pt"), map_location="cpu")
    _model = VelocityTransformer(n_genres=len(_gv) + 1)
    _model.load_state_dict(_ck["best_model"])
    _model.eval()

    def predict(feats):
        t = build_split_tensors(feats, _gv, _sc["bpm_mean"], _sc["bpm_std"])
        with torch.no_grad():
            y = _model(t["voice_idx"], t["genre_idx"], t["num_feats"], t["pad_mask"])
        return scatter_predictions(t["row_idx"], y, t["pad_mask"], len(feats))

else:
    raise ValueError(f"unknown MODEL {MODEL!r}")

print(f"loaded {MODEL}")

## The `audition` helper

Picks a sample, predicts its velocities, and shows the metrics, three audio players
(predicted / original / flat), and three shared-scale piano rolls.

In [ ]:
def audition(sample=None, split=SPLIT, beat_type=None, flat=True, rolls=True):
    """Audition one performance. `sample`: int index into the split, or None=random."""
    pool = csv[csv.split == split]
    if beat_type:
        pool = pool[pool.beat_type == beat_type]
    pool = pool.reset_index(drop=True)
    i = np.random.randint(len(pool)) if sample is None else int(sample) % len(pool)
    meta = pool.iloc[i].to_dict()
    path = os.path.join(BASE, meta["midi_filename"])

    na = load_note_array(path)
    feats = build_note_features(na, meta)
    order = np.argsort(na["onset_sec"], kind="stable")   # feats row i <-> na[order][i]
    true_vel = na["velocity"][order].astype(float)
    pred = np.clip(np.rint(predict(feats)), 0, 127)

    display(Markdown(
        f"**[{split} #{i}] {meta['id']}** &middot; style=`{meta['style']}` "
        f"bpm={meta['bpm']} time_sig={meta['time_signature']} beat={meta['beat_type']} "
        f"&middot; {len(na)} notes<br>"
        f"`{MODEL}` &mdash; per-track MAE **{mae(true_vel, pred):.2f}**, "
        f"RMSE {rmse(true_vel, pred):.2f}, pred std {pred.std():.1f} (true {true_vel.std():.1f})"))

    na_pred = na[order].copy()
    na_pred["velocity"] = pred.astype(na["velocity"].dtype)
    na_flat = na[order].copy()
    na_flat["velocity"] = np.full(len(na_flat), 80, dtype=na["velocity"].dtype)

    display(Markdown("**▶️ Original**"));  display(play_midi_file(path, is_drums=True))
    display(Markdown("**▶️ Predicted**")); display(play_midi_notes(na_pred, is_drums=True))
    if flat:
        display(Markdown("**▶️ Flat (constant velocity)**")); display(play_midi_notes(na_flat, is_drums=True))

    if rolls:
        # colour intensity = velocity; shared 0-127 scale makes the three comparable
        display(Markdown("**🎹 Original**"));  drums_roll(na[order], figsize=(15, 4), vmin=0, vmax=1)
        display(Markdown("**🎹 Predicted**")); drums_roll(na_pred, figsize=(15, 4), vmin=0, vmax=1)
        if flat:
            display(Markdown("**🎹 Flat**")); drums_roll(na_flat, figsize=(15, 4), vmin=0, vmax=1)
    return meta

## Audition a sample

**Change the number and re-run this cell** to hear/see a different performance.

In [ ]:
audition(0)

A random sample each time you run this cell:

In [ ]:
audition()

## Interactive picker (slider)

Drag to scrub through the split. (Requires `ipywidgets`; falls back to a hint if absent.)

In [ ]:
try:
    from ipywidgets import interact, IntSlider
    _n = int((csv.split == SPLIT).sum())
    interact(lambda sample: audition(sample),
             sample=IntSlider(value=0, min=0, max=_n - 1, step=1, description='sample'))
except ImportError:
    print("ipywidgets not installed — call audition(N) directly instead")